In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
y = train["PitNextLap"].astype(int).values

oof_002 = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260501_002_cat_strict_raw_baseline.npy"
)
oof_006b = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_006b_cat_original_prior_no_year_prior_min.npy"
)
oof_006c = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_006c_cat_original_prior_no_count_min.npy"
)
oof_007 = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_007_realmlp_shared001_min.npy"
)
oof_008 = np.load(
    "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_exp_20260503_008_realmlp_shared001_plus_006b_orig_prior.npy"
)

def report(name, oof):
    print(name)
    print("  AUC:", roc_auc_score(y, oof))
    print()

report("002_cat_raw", oof_002)
report("006b_cat_original_prior_no_year", oof_006b)
report("006c_cat_original_prior_no_count", oof_006c)
report("007_realmlp_shared001_min", oof_007)
report("008_realmlp_shared001_plus_orig_prior", oof_008)

pairs = [
    ("006b", oof_006b, "007", oof_007),
    ("006b", oof_006b, "008", oof_008),
    ("007", oof_007, "008", oof_008),
    ("006c", oof_006c, "008", oof_008),
]

print("\nCorrelation:")
for a_name, a, b_name, b in pairs:
    pearson = np.corrcoef(a, b)[0, 1]
    spearman = spearmanr(a, b).correlation
    print(f"{a_name} vs {b_name}: pearson={pearson:.6f}, spearman={spearman:.6f}")

blend_candidates = {
    "avg_007_008": (oof_007 + oof_008) / 2,
    "avg_006b_007": (oof_006b + oof_007) / 2,
    "avg_006b_008": (oof_006b + oof_008) / 2,
    "avg_006b_007_008": (oof_006b + oof_007 + oof_008) / 3,
    "avg_006b_006c_007_008": (oof_006b + oof_006c + oof_007 + oof_008) / 4,
}

print("\nBlend probe:")
for name, o in blend_candidates.items():
    print(name, roc_auc_score(y, o))

002_cat_raw
  AUC: 0.9485020378244504

006b_cat_original_prior_no_year
  AUC: 0.9487943937294769

006c_cat_original_prior_no_count
  AUC: 0.9485750913142212

007_realmlp_shared001_min
  AUC: 0.9532701289015783

008_realmlp_shared001_plus_orig_prior
  AUC: 0.9530389769769412


Correlation:
006b vs 007: pearson=0.974165, spearman=0.963871
006b vs 008: pearson=0.976387, spearman=0.964499
007 vs 008: pearson=0.994703, spearman=0.991720
006c vs 008: pearson=0.975489, spearman=0.963713

Blend probe:
avg_007_008 0.9535034403208038
avg_006b_007 0.9528554748055666
avg_006b_008 0.95262257316863
avg_006b_007_008 0.95339881683978
avg_006b_006c_007_008 0.952838112601165
